# Submission Akhir BMLP - Klasifikasi
**Nama:** Rava Amesta

Notebook ini membangun model klasifikasi untuk memprediksi label cluster (`Target`) hasil dari notebook clustering (`data_clustering.csv`).

## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

import joblib


## 2. Memuat Dataset
Dataset yang digunakan adalah hasil export dari notebook clustering, yaitu `data_clustering.csv`, yang sudah memiliki kolom label `Target` (hasil clustering).

In [ ]:
df = pd.read_csv("data_clustering.csv")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## 3. Data Splitting
Memisahkan fitur (`X`) dan label (`y`), lalu membagi data menjadi data latih dan data uji menggunakan `train_test_split()`.

In [ ]:
X = df.drop(columns=["Target"])
y = df["Target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Jumlah data latih :", X_train.shape[0])
print("Jumlah data uji   :", X_test.shape[0])


## 4. Membangun Model Klasifikasi

### 4.1 Decision Tree (Model Utama)

In [ ]:
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)

y_pred_dt = dt_model.predict(X_test)

print("Akurasi  :", accuracy_score(y_test, y_pred_dt))
print("Precision:", precision_score(y_test, y_pred_dt, average="weighted"))
print("Recall   :", recall_score(y_test, y_pred_dt, average="weighted"))
print("F1-Score :", f1_score(y_test, y_pred_dt, average="weighted"))
print()
print(classification_report(y_test, y_pred_dt))


### 4.2 Menyimpan Model Decision Tree

In [ ]:
joblib.dump(dt_model, "decision_tree_model.h5")
print("Model Decision Tree berhasil disimpan sebagai decision_tree_model.h5")

## 5. Eksplorasi Model Lain (Opsional)
Mencoba beberapa algoritma klasifikasi lain sebagai perbandingan terhadap Decision Tree.

In [ ]:
explore_models = {
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "RandomForest": RandomForestClassifier(random_state=42),
    "LogisticRegression": LogisticRegression(max_iter=1000),
}

explore_results = {}

for name, clf in explore_models.items():
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="weighted")
    explore_results[name] = {"model": clf, "accuracy": acc, "f1_score": f1}
    print(f"{name:<20} | Akurasi: {acc:.4f} | F1-Score: {f1:.4f}")

best_model_name = max(explore_results, key=lambda k: explore_results[k]["f1_score"])
best_explore_model = explore_results[best_model_name]["model"]
print("\nModel eksplorasi terbaik:", best_model_name)


In [ ]:
joblib.dump(best_explore_model, "explore__classification.h5")
print("Model eksplorasi terbaik disimpan sebagai explore__classification.h5")

## 6. Hyperparameter Tuning (Opsional)
Melakukan tuning pada model Decision Tree menggunakan `GridSearchCV` untuk mencari kombinasi parameter terbaik.

In [ ]:
param_grid = {
    "max_depth": [3, 5, 7, 10, None],
    "min_samples_split": [2, 5, 10],
    "criterion": ["gini", "entropy"],
}

grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring="f1_weighted",
    n_jobs=-1,
)

grid_search.fit(X_train, y_train)

print("Parameter terbaik:", grid_search.best_params_)
print("Best CV F1-Score  :", grid_search.best_score_)

tuned_model = grid_search.best_estimator_
y_pred_tuned = tuned_model.predict(X_test)

print("\nHasil pada data uji setelah tuning:")
print(classification_report(y_test, y_pred_tuned))


In [ ]:
joblib.dump(tuned_model, "tuning_classification.h5")
print("Model hasil tuning disimpan sebagai tuning_classification.h5")

## 7. Kesimpulan
Model Decision Tree utama (`decision_tree_model.h5`) berhasil dilatih untuk memprediksi label cluster (`Target`) berdasarkan fitur-fitur transaksi nasabah, dengan performa yang dievaluasi menggunakan accuracy, precision, recall, dan f1-score. Model eksplorasi dan hasil tuning hyperparameter turut disertakan sebagai perbandingan tambahan (opsional).